# S6E9 · Fold-safe ensembling to the honest ceiling (nested OOF 0.94642)

**What this notebook shows, with code you can fork:**
1. one frozen outer split shared with the public OOF libraries,
2. **nested** target encoding of the income memory (a row never sees its own label; the scored fold is never seen),
3. early stopping on a slice of outer-train, never on the scored fold,
4. averaging each model family over **three outer splits × three seeds**,
5. a hill-climb blend whose own score is **out of fold** (weights fit on 9 folds, scored on the 10th),
6. a paired bootstrap at private-leaderboard size to check whether a gain would survive the private split.

The package, tests, figures and every number are in the open-source repository: [https://github.com/Agnuxo1/s6e9-honest-ceiling](https://github.com/Agnuxo1/s6e9-honest-ceiling) (Apache-2.0).
The companion notebook, *Darwin Cage on the residual*, asks what is left after this model.

> Honest framing: this pipeline does not beat the public pack. It reaches **0.946420 nested OOF**, within the sampling
> error of its own oracle bound (0.94660 ± 0.00028). Its value is that every number is out of fold, so it tells you
> what a blend is really worth before the private leaderboard does.

## 1 · Setup
The wheel of the `honest_ceiling` package ships in the companion dataset, so the notebook runs offline.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def find(name, roots=("/kaggle/input", os.environ.get("HC_LOCAL", "."))):
    """First file called `name` under the given roots (handles legacy and namespaced Kaggle input paths)."""
    for r in roots:
        hits = sorted(glob.glob(f"{r}/**/{name}", recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(name)

try:
    import honest_ceiling  # noqa: F401
except ImportError:  # the wheel ships inside the companion dataset; no internet needed
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find("honest_ceiling-1.0.0-py3-none-any.whl"))], check=True)
import honest_ceiling
from honest_ceiling.data import load, outer_folds
from honest_ceiling.blend import rank01, fast_auc, nested_hillclimb, paired_bootstrap

COMP = find("train.csv").parent
LIB = find("oof_legs.csv").parent            # companion dataset: our OOF/test library, folds, results
print("honest_ceiling", honest_ceiling.__version__, "| competition data:", COMP, "| library:", LIB)
train, test, y = load(COMP)
fold = outer_folds(y)
print(f"train {train.shape}  test {test.shape}  positives {y.mean():.4f}")

plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#e6e5e1",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
                     "axes.titleweight": "semibold", "axes.titlelocation": "left", "font.size": 10})
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#b9b8b2"

## 2 · The shared split
`StratifiedKFold(10, shuffle=True, random_state=42)` over `train.csv` in id order. It is the split of the public CC0
six-view OOF library, so our out-of-fold vectors stack row for row with theirs.

In [ ]:
lib = pd.read_csv(LIB / "oof_legs.csv")
assert (lib["id"].to_numpy() == train["id"].to_numpy()).all()
assert (lib["fold_s42"].to_numpy() == fold).all(), "fold ids differ from the published library"
print("fold ids identical to the published library for all", len(fold), "rows")
print(pd.Series(fold).value_counts().sort_index().to_dict())

## 3 · Features and nested target encoding
`build_base` reads no label: the generator's purchase and range-anxiety recipes (reconstructed by Chris Deotte),
income digits and remainders, frequencies, and lookups into the CC0 original table. `fold_matrices` adds the
supervised columns for one outer fold, cross-fitted inside outer-train.

A quick leakage check first: with labels shuffled, a leaky encoder would still score well on its own fit rows.

In [ ]:
from honest_ceiling.features import build_base, fold_matrices, nested_target_encode
try:
    ORIG = find("EV_Adoption_and_Range_Anxiety_Dataset.csv")
except FileNotFoundError:
    ORIG = None
X, K = build_base(train, test, ORIG)
print("unsupervised features:", X.shape[1], "| integer keys:", K.shape[1])

rng = np.random.default_rng(0)
y_shuffled = rng.permutation(y)
enc_fit, _ = nested_target_encode(K.iloc[:len(train)], y_shuffled, [K.iloc[:5]], seed=0, spec={"k_income_exact": [1.0]})
print("AUC of the income encoding against shuffled labels (0.5 = no leakage):",
      round(fast_auc(y_shuffled, enc_fit["te_income_exact_a1"].to_numpy()), 4))

## 4 · One fold-safe leg, live
XGBoost on one seed and the shared split. Early stopping uses a stratified 10 % slice of outer-train. This is one of
the 18 GBDT runs behind the final blend; the full set is loaded in the next section.

In [ ]:
from honest_ceiling.train import train_leg
t0 = time.time()
rep = train_leg(X, K, y, len(train), len(test), "leg_xgb_s42", model="xgb", seed=42, outer_seed=42)
print(f"OOF AUC {rep['oof_auc']:.6f} in {time.time()-t0:.0f} s")
published = json.load(open(LIB / "legs.json"))["final_xgb_s42"]
print("published run   ", round(published["oof_auc"], 6))
print("per-fold difference vs published (GPU histograms are not bit-exact):",
      np.round(np.array(rep["fold_aucs"]) - np.array(published["fold_aucs"]), 6))

## 5 · Averaging over outer splits
A single run's OOF depends on which rows its encoders and early stopping happened to see. Averaging the same model
over different outer splits (seeds 42, 7, 13) and seeds (42, 101, 2026) in rank space removes part of that variance.

In [ ]:
legs = json.load(open(LIB / "legs.json"))
single = np.mean([v["oof_auc"] for k, v in legs.items() if k.startswith("final_xgb")])
print(f"XGBoost: mean single-run OOF {single:.6f}  ->  9-run rank average {fast_auc(y, lib['xgb_x9'].to_numpy()):.6f}")
single = np.mean([v["oof_auc"] for k, v in legs.items() if k.startswith("final_lgbm")])
print(f"LightGBM: mean single-run OOF {single:.6f}  ->  9-run rank average {fast_auc(y, lib['lgbm_x9'].to_numpy()):.6f}")

## 6 · Nested hill-climb
Streams: our legs plus the public CC0 six-view library (views A-F and RealMLP-G, same split). Weights are fit on
9 folds and scored on the 10th, for every fold, so the blend's AUC is itself out of fold.
Attach [megayak/s6e9-six-feature-views-oof-library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library) to reproduce this cell.

In [ ]:
SIX = find("oof_six_views.csv").parent
so = pd.read_csv(SIX / "oof_six_views.csv").set_index("id").loc[train["id"]]
go = pd.read_csv(SIX / "oof_realmlp_g.csv").set_index("id").loc[train["id"]]
names = ["xgb_x9", "lgbm_x9", "logreg_x1", "mlp_x2", "realmlp_x1", "ladderxgb_x1"]
O = [lib[n].to_numpy(float) for n in names]
for c in [c for c in so.columns if c[:2] in ("A_", "B_", "C_", "D_", "E_", "F_")]:
    names.append("six:" + c[0]); O.append(so[c].to_numpy(float))
names.append("six:G_realmlp"); O.append(go["G_realmlp_10fold"].to_numpy(float))
O = np.column_stack([rank01(o) for o in O])
t0 = time.time()
res = nested_hillclimb(O, y, fold)
print(f"nested OOF AUC {res['nested_auc']:.6f}   (in-sample {res['in_sample_auc']:.6f})   {time.time()-t0:.0f} s")
print(pd.Series(res["weights"], index=names).round(3).sort_values(ascending=False).to_string())

In [ ]:
ens = so["ensemble"].to_numpy(float)
d = np.array([fast_auc(y[fold == k], res["nested_scores"][fold == k]) - fast_auc(y[fold == k], ens[fold == k]) for k in range(10)])
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(range(10), d * 1e5, color=BLUE, width=0.62)
for i, v in enumerate(d * 1e5):
    ax.text(i, v + 0.2, f"+{v:.1f}" if v >= 0 else f"{v:.1f}", ha="center", fontsize=9, color="#52514e")
ax.set_xticks(range(10), [f"fold {k}" for k in range(10)]); ax.set_ylabel("AUC gain × 1e-5")
ax.set_title(f"Nested blend vs public six-view ensemble: {int((d > 0).sum())} of 10 folds")
plt.show()

## 7 · Would the gain survive the private leaderboard?
The private split has 229,257 rows. Scoring both vectors on the *same* random subsamples of that size is how the
private leaderboard compares two teams.

In [ ]:
for n, label in ((229_257, "private-size"), (57_314, "public-size")):
    pb = paired_bootstrap(y, res["nested_scores"], ens, n=n, reps=40, seed=1)
    print(f"{label:12s} gain {pb['mean']:+.6f} ± {pb['std']:.6f}   P(better) {pb['p_a_better']:.2f}")

## 8 · Submission
The test predictions use the same weights on the test ranks of every stream.

In [ ]:
st = pd.read_csv(SIX / "test_six_views.csv").set_index("id").loc[test["id"]]
gt = pd.read_csv(SIX / "test_realmlp_g.csv").set_index("id").loc[test["id"]]
tl = pd.read_csv(LIB / "test_legs.csv")
T = [tl[n].to_numpy(float) for n in names[:6]]
T += [st[c].to_numpy(float) for c in [c for c in so.columns if c[:2] in ("A_", "B_", "C_", "D_", "E_", "F_")]]
T += [gt["G_realmlp_10fold"].to_numpy(float)]
T = np.column_stack([rank01(t) for t in T])
sub = pd.DataFrame({"id": test["id"], "Will_Buy_EV": rank01(T @ res["weights"])})
sub.to_csv("submission.csv", index=False)
print(sub.head(), "\nSpearman with the published v4 file:", round(sub["Will_Buy_EV"].corr(tl["blend_v4"], method="spearman"), 6))

## What to take from it

* Every supervised number in a blend should be out of fold, **including the blend weights**. The nested score here
  is 0.0000047 below the in-sample one: small, but it is the honest one.
* Averaging a model over several outer splits is one of the few gains that remained at this saturation.
* A gain is only a gain if it survives a paired comparison at the size of the split that will judge it.

Code, tests, the full results and an open list of research directions: [https://github.com/Agnuxo1/s6e9-honest-ceiling](https://github.com/Agnuxo1/s6e9-honest-ceiling).
If you find an honest way above this ceiling, open an issue there: it will be added with credit.